In [5]:
# BRFSS 2024: Milestone 1 - Data Acquisition & Understanding

import pandas as pd
import numpy as np

# M1.T1: Load BRFSS 2024 dataset

df = pd.read_parquet('brfss_clean_processed.parquet')
print("Dataset loaded. Shape:", df.shape)

# M1.T2: Auto-map health, demographic, and behavioral variables

health_patterns = ['BMI', 'ASTH', 'MICHD', 'RFHLTH', 'PHYS14D', 'MENT14D', 'HLTHPL2', 'DENVST3', 'DIABETES', 'HYPERTENSION', 'CHOLESTEROL']
demographic_patterns = ['AGE', 'SEX', 'RACE', 'EDU', 'INCOME']
behavioral_patterns = ['EXERCISE', 'SMOKER', 'ALCOHOL', 'FRUIT', 'VEG', 'SODA']

def select_columns(df, patterns):
    cols = []
    for p in patterns:
        cols += [c for c in df.columns if p.lower() in c.lower()]
    return sorted(list(set(cols)))

health_vars = select_columns(df, health_patterns)
demo_vars = select_columns(df, demographic_patterns)
behavior_vars = select_columns(df, behavioral_patterns)

print("Health Variables Detected:", health_vars)
print("Demographic Variables Detected:", demo_vars)
print("Behavioral Variables Detected:", behavior_vars)

# M1.T3: Define obesity target variable

if 'BMI' in df.columns:
    df['obesity'] = (df['BMI'] >= 30).astype(int)
elif any('RFBMI' in c for c in df.columns):
    bmi_col = [c for c in df.columns if 'RFBMI' in c][0]
    df['obesity'] = (df[bmi_col] >= 30).astype(int)
else:
    raise ValueError("No BMI variable detected. Please verify dataset.")

print("Obesity target variable created. Distribution:")
print(df['obesity'].value_counts())

# M1.T3: Cross-check health predictors for completeness

df_health = df[health_vars].copy()
print("Health predictor subset info:")
print(df_health.info())
print("\nMissing values per health predictor:")
print(df_health.isnull().sum())

# Save preliminary subset for team review
mapped_vars = health_vars + demo_vars + behavior_vars + ['obesity']
df_mapped = df[mapped_vars].copy()
df_mapped.to_parquet('brfss_2024_mapped_vars.parquet', index=False)
print("Mapped variables saved to 'brfss_2024_mapped_vars.parquet'")

Dataset loaded. Shape: (409415, 305)
Health Variables Detected: ['ASTHMA3', 'ASTHNOW', 'BMI', 'BMI_raw', 'CASTHDX2', 'CASTHNO2', '_ASTHMS1', '_BMI5CAT', '_CASTHM1', '_DENVST3', '_HLTHPL2', '_LTASTH1', '_MENT14D', '_MICHD', '_PHYS14D', '_RFBMI5', '_RFHLTH']
Demographic Variables Detected: ['ACEHVSEX', 'Age_group', 'CAGEG', 'CELLSEX3', 'CNCRAGE', 'DIABAGE4', 'DIABEDU1', 'EDUCA', 'Education_level', 'HADSEX', 'INCOME3', 'Income_Cat', 'LANDSEX3', 'SEXVAR', 'Sex', '_AGE65YR', '_AGE80', '_AGEG5YR', '_CRACE1', '_IMPRACE', '_LCSAGE', '_MRACE1', '_RACE', '_RACEG21', '_RACEGR3', '_RACEPRV']
Behavioral Variables Detected: ['Smoker', '_SMOKER3']
Obesity target variable created. Distribution:
obesity
0    409415
Name: count, dtype: int64
Health predictor subset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 409415 entries, 0 to 409414
Data columns (total 17 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   ASTHMA3   409414 non-null  float64
 1   